In [6]:
from rsw_ai.backend.import_nadinpethiyagoda_vehicle_dataset_for_yolo_to_VisionYoloDataset import (
    import_nadinpethiyagoda_vehicle_dataset_for_yolo_to_VisionYoloDataset,
)
from kaggle.api.kaggle_api_extended import KaggleApi
from rsw_ai.enum.DatasetRepositoryLayout import DatasetRepositoryLayout
from rsw_ai.model.YoloExporter import YoloExporter
from rsw_ai.backend.preprocess_images import preprocess_images
from pathlib import Path

from ultralytics import YOLO


In [7]:
# ====================================
# Initiate Variable
# ====================================

KAGGLE_DOWNLOADED_PATH = "outputs/datasets/downloads/kaggle/"  # The main directory after downloading from the Kaggle dataset.

dataset_repository_layout = (
    DatasetRepositoryLayout.NADINPETHIYAGODA_VEHICLE_DATASET_FOR_YOLO
)
print("dataset_repository_layout : ", dataset_repository_layout)

DIR_DOWNLOADED = (
    "../"  # backward one dir
    + KAGGLE_DOWNLOADED_PATH
    + dataset_repository_layout.label
    + "/"
)
print("DIR_DOWNLOADED : ", DIR_DOWNLOADED)


DIR_VISION_YOLO_DATASET_OUTPUT = (
    "outputs/datasets/yolo/"  # Output directory of the processed YOLO dataset
)

DATASET_OUTPUT_PATH = (
    "../"  # backward one dir
    + DIR_VISION_YOLO_DATASET_OUTPUT
    + dataset_repository_layout.label
    + "/"
)
print("DATASET_OUTPUT_PATH : ", DATASET_OUTPUT_PATH)


DIR_EXPERIMENT_OUTPUT = "../outputs/experiments/yolo/"

EXPERIMENT_OUTPUT_PATH = Path(DIR_EXPERIMENT_OUTPUT).resolve()
print("EXPERIMENT_OUTPUT_PATH : ", EXPERIMENT_OUTPUT_PATH)


model_path = Path("../outputs/models/yolo/yolo11n.pt").resolve()


DATASET_YAML_PATH = Path(DATASET_OUTPUT_PATH + "dataset.yaml").resolve()
print("DATASET_YAML_PATH : ", DATASET_YAML_PATH)


# ====================================
# Initiate Variable (END)
# ====================================


dataset_repository_layout :  DatasetRepositoryLayout.NADINPETHIYAGODA_VEHICLE_DATASET_FOR_YOLO
DIR_DOWNLOADED :  ../outputs/datasets/downloads/kaggle/nadinpethiyagoda_vehicle_dataset_for_yolo/
DATASET_OUTPUT_PATH :  ../outputs/datasets/yolo/nadinpethiyagoda_vehicle_dataset_for_yolo/
EXPERIMENT_OUTPUT_PATH :  C:\Users\vsgm0\Downloads\AI\temp\qp_ai_vision\outputs\experiments\yolo
DATASET_YAML_PATH :  C:\Users\vsgm0\Downloads\AI\temp\qp_ai_vision\outputs\datasets\yolo\nadinpethiyagoda_vehicle_dataset_for_yolo\dataset.yaml


In [ ]:
# ====================================
#  Logic - Download dataset
# ====================================
api = KaggleApi()
api.authenticate()

api.dataset_download_files(
    dataset_repository_layout.dataset_name,
    path=DIR_DOWNLOADED,
    unzip=True,
)

print(DIR_DOWNLOADED)

In [2]:
# ====================================
#  Logic - preprocessing
# ====================================

preprocess_images(
    input_dir="../outputs/datasets/downloads/kaggle/nadinpethiyagoda_vehicle_dataset_for_yolo/vehicle dataset",
    output_dir="../outputs/datasets/preprocessed/nadinpethiyagoda_vehicle_dataset_for_yolo/vehicle dataset",
)

In [3]:
# ====================================
#  Logic - import dataset
# ====================================

vision_yolo_dataset = import_nadinpethiyagoda_vehicle_dataset_for_yolo_to_VisionYoloDataset(
    "../outputs/datasets/preprocessed/nadinpethiyagoda_vehicle_dataset_for_yolo/vehicle dataset"
)

for split in vision_yolo_dataset.splits:
    print(split.name)
    for sample in split.samples:
        print(sample)

train
Sample(input='..\\outputs\\datasets\\preprocessed\\nadinpethiyagoda_vehicle_dataset_for_yolo\\vehicle dataset\\train\\images\\00043_GMC Savana Van 2012.jpg', target=[YoloAnnotation(class_id=5, center_x=0.499161, center_y=0.503597, width=0.971477, height=0.767386)])
Sample(input='..\\outputs\\datasets\\preprocessed\\nadinpethiyagoda_vehicle_dataset_for_yolo\\vehicle dataset\\train\\images\\00077.jpg', target=[YoloAnnotation(class_id=0, center_x=0.49634, center_y=0.518811, width=0.97241, height=0.880632)])
Sample(input='..\\outputs\\datasets\\preprocessed\\nadinpethiyagoda_vehicle_dataset_for_yolo\\vehicle dataset\\train\\images\\00116.jpg', target=[YoloAnnotation(class_id=0, center_x=0.505089, center_y=0.526423, width=0.676845, height=0.569106)])
Sample(input='..\\outputs\\datasets\\preprocessed\\nadinpethiyagoda_vehicle_dataset_for_yolo\\vehicle dataset\\train\\images\\00175_Ford E-Series Wagon Van 2012.jpg', target=[YoloAnnotation(class_id=5, center_x=0.503125, center_y=0.578799

In [8]:
# ====================================
#  Logic - export dataset
# ====================================

yolo_exporter = YoloExporter()
 
yolo_exporter.export_dataset(DATASET_OUTPUT_PATH, vision_yolo_dataset)


In [9]:
# ====================================
# Logic - Load model
# ====================================

model = YOLO("yolo11n.yaml")

print(model)

YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C3k2(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=

In [10]:
# ====================================
# Logic - Train
# ====================================
print("DATASET_YAML_PATH : ", DATASET_YAML_PATH)
print("DIR_EXPERIMENT_OUTPUT : ", DIR_EXPERIMENT_OUTPUT)

results = model.train(
    data=str(DATASET_YAML_PATH),

    project=str(DIR_EXPERIMENT_OUTPUT),

    name=dataset_repository_layout.label,

    epochs=30,
    imgsz=320,
    batch=2,
    device="cpu",
    workers=0,
)

print(results)

DATASET_YAML_PATH :  C:\Users\vsgm0\Downloads\AI\temp\qp_ai_vision\outputs\datasets\yolo\nadinpethiyagoda_vehicle_dataset_for_yolo\dataset.yaml
DIR_EXPERIMENT_OUTPUT :  ../outputs/experiments/yolo/
New https://pypi.org/project/ultralytics/8.4.135 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.115  Python-3.11.15 torch-2.12.1+cpu CPU (AMD Ryzen 5 7520U with Radeon Graphics)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\vsgm0\Downloads\AI\temp\qp_ai_vision\outputs\datasets\yolo\nadinpethiyagoda_vehicle_dataset_for_yolo\dataset.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=F

In [ ]:
from ultralytics import YOLO

model = YOLO('C:/Users/vsgm0/Downloads/AI/temp/qp_ai_vision/notebooks/runs/detect/outputs/experiments/yolo/nadinpethiyagoda_vehicle_dataset_for_yolo-2/weights/best.pt')

print(dir(model))

if hasattr(model, 'ckpt'):
    print(model.ckpt)

if hasattr(model.model, 'ckpt'):
    print(model.model.ckpt)
    

['T_destination', '__annotations__', '__call__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_apply', '_backward_hooks', '_backward_pre_hooks', '_buffers', '_call_impl', '_check_is_pytorch_model', '_compiled_call_impl', '_forward_hooks', '_forward_hooks_always_called', '_forward_hooks_with_kwargs', '_forward_pre_hooks', '_forward_pre_hooks_with_kwargs', '_get_backward_hooks', '_get_backward_pre_hooks', '_get_name', '_is_full_backward_hook', '_load', '_load_from_state_dict', '_load_state_dict_post_hooks', '_load_state_dict_pre_hooks', '_maybe_warn_non_full_backward_hook', '_modules', '_named_members', '_new', '_non_persistent_buffers_set', '_par